# Job/Internship Application Assistant Agent
An AI agent that parses resumes and job postings, generates tailored cover letters
with skill-gap analysis, and tracks applications over time using natural language —
built for Kaggle's AI Agents Intensive Vibe Coding Capstone.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but

In [3]:
from kaggle_secrets import UserSecretsClient
from google import genai

# Load the secret you saved earlier
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")

# Create the Gemini client
client = genai.Client(api_key=api_key)

# Test call
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello and confirm you're working!"
)

print(response.text)

Hello! Yes, I'm here and working perfectly. How can I help you today?


## Phase 1: Resume + Job Matching

In [9]:
resume_text = """
PASTE YOUR RESUME TEXT HERE.

Example structure:
Name: Mujeeb Ullah Khan
Education: B.Sc. in information systems, Raden Intan State Islamic University of Lampung, 6th semester
Skills: Python, JavaScript, HTML/CSS, AI/ML, Firebase, IoT systems
Experience: SEO writer internship internships, system developer projects, IOT smart Door Project,
Projects: SobrCafe POS system, IoT/AI energy efficiency monitoring thesis, ICSSF 2026 conference paper
"""

In [5]:
job_posting_text = """
Digital, AI and Innovation Internship: Global Call for 2026
Home Based
Trending
Job Info
Job Identification
33001
Posting Date
03/31/2026, 03:42 AM
Apply Before
10/01/2026, 10:59 AM
Job Schedule
Full time
Locations
 Home Based
Agency
UNDP
Vacancy Type
Internship Programme
Practice Area
Innovation
Bureau
Bureau for Policy and Programme Support
Contract Duration
3 months
Required Languages
Fluency in English
Job Description
BACKGROUND

 
Diversity, Equity and Inclusion are core principles at UNDP:  we value diversity as an expression of the multiplicity of nations and cultures where we operate, we foster inclusion as a way of ensuring all personnel are empowered to contribute to our mission, and we ensure equity and fairness in all our actions. Taking a ‘leave no one behind’ approach to our diversity efforts means increasing representation of underserved populations. People who identify as belonging to marginalized or excluded populations are strongly encouraged to apply. Learn more about working at UNDP including our values and inspiring stories.

 
UNDP does not tolerate sexual exploitation and abuse, any kind of harassment, including sexual harassment, and discrimination. All selected candidates will, therefore, undergo rigorous reference and background checks.
 
The United Nations Development Programme (UNDP) is the development arm of the United Nations. Present in 170 countries with over 17,000 people, UNDP works on the world’s biggest problems: extreme poverty, climate change, good governance, renewable energy, crisis prevention, and women’s empowerment, among others.  

Global development is increasingly shaped by interconnected crises, rapid technological change, and growing uncertainty. At the same time, digital technologies and artificial intelligence are transforming economies, governance systems, and public services, creating new opportunities to accelerate sustainable development when applied in inclusive and responsible ways. 

As the development arm of the United Nations, UNDP serves as the knowledge frontier organization for sustainable development and acts as the integrator for collective action to advance the Sustainable Development Goals (SDGs). Working in over 170 countries and territories, UNDP’s policy and programme work spans global, regional, and national levels, linking deep local knowledge with cutting-edge global expertise in support of the UNDP Strategic Plan 2026–2029. Within this institutional framework, the Bureau for Policy and Programme Support (BPPS) leads the development of corporate policies, technical guidance, and operational frameworks to advance UNDP’s mandate. 

The Strategic Plan and the organization’s Digital Strategy recognize digitalization, data, and innovation as key accelerators for achieving development outcomes. These frameworks position digital and AI as foundational enablers for stronger institutions, evidence-based decision-making, and more effective service delivery.  

Within this institutional context, the Digital, AI and Innovation (DAI) Hub serves as UNDP’s central platform for advancing systems transformation, inclusive digitalization, and innovation across its development work. The Hub brings together expertise in systems transformation, digital public infrastructure, data and artificial intelligence, bottom-up innovation, capacity development, and digital product design. Through an integrated approach, the Hub supports countries and UNDP teams to design integrated, scalable solutions that respond to both immediate development needs and long-term structural challenges. 

By combining global expertise with field-based insight and strong partnerships, the DAI Hub contributes to UNDP’s mandate to deliver more agile, inclusive, and future-ready development solutions. 

 

Please note that the start date for the internship is flexible, and needs-based. We will be receiving applications on a rolling basis until September 30th, 2026, and successful applicants can expect to hear back any time between April and the end of 2026.  

Please read the instructions carefully to complete your application – applications are to be submitted both via the UNDP Vacancy Link AND the Airtable link: https://airtable.com/appeKAJ7quPABAfoc/pagj4yv4UGVuveqXx/form  

 

 
DUTIES AND RESPONSIBILITIES

We are looking for motivated interns who can support our work in any of the following areas listed below:  

 

Systems Transformation & Innovation

Conduct research and draft analytical content on systems transformation, innovation, and emerging policy approaches 
Contribute to horizon scanning and synthesis of global and regional trends relevant to policy innovation and system change 
Contribute to the design, documentation, and refinement of innovative policy and programme approaches 
Design, test, and document innovative policy and programme approaches aimed at system-level change 
Collect, curate, and disseminate knowledge products and thought leadership on innovation and systems transformation to align regional efforts and scale innovation
 

Programme Delivery & Capacity Building

Contribute to the delivery of digital, data, and AI initiatives with UNDP Country Offices 
Develop capacity-building materials on digital, data, and AI for governments and partners 
Coordinate activities related to national digital, data, and AI strategies 
Draft practical toolkits, guidance notes, and implementation resources for Country Offices 
Organize and deliver webinars, trainings, and learning sessions
 

Data Policy & Governance 
 

Conduct research on data governance frameworks, data protection, and responsible data use 
Contribute to advisory on development of national data strategies and data governance frameworks 
Draft policy briefs, background notes, and analytical summaries on data-related topics 
Track global and multilateral processes related to data and digital governance 
Contribute to UNDP positioning and advocacy on data policy and digital rights
 

Data Analytics & Insights 

Collect, clean, and analyze quantitative and qualitative datasets 
Develop dashboards, trackers, and data visualizations for monitoring and reporting 
Analyze progress of Digital Strategy implementation using data-driven approaches 
Contribute to monitoring, evaluation, and reporting processes 
Translate data findings into concise, actionable insights for internal use
 

Partnerships & Strategic Communications 
 

Research potential new partners including technology companies, foundations, non-profits, alliances, etc., and help identify new opportunities and areas of collaboration 
Contribute to strategic communications, outreach to and engagement of partners, and subsequent partnership development, including drafting concept notes, communications materials, proposals, pitch decks, project documents, memorandum of understanding (MoU), outreach materials and other documents. 
Take part in engagement in global, multilateral and UN interagency processes, including through drafting statements and speeches, attending meetings at the UN, and helping to organize meetings and events. 
Assist in UNDP positioning and thought leadership on digital, AI and innovation including through conducting research, compiling information on, and analyzing key technology and digital issues, particularly related to digital development, transformation and capacity-building 
Develop strategic communications by planning and producing compelling content for communications channels like social media, newsletters, blogs and internal UNDP platforms 
Contribute to the research and mapping of key media outlets, influential potential content partners, and trending topics in digital & technology
 

Process Efficiency & Resource Management 

Track project workplans, milestones, and deliverables for DAI Hub projects 
Contribute to planning for HR, procurement, and resourcing processes in line with UNDP policies and guidelines 
Develop SOPs, internal guidance, trackers, and dashboards 
Contribute to planning, monitoring, and reporting of operational and resourcing plans 
Identify operational inefficiencies and propose process improvements
 

Product Design, Build, and Management (including AI tools) 

Participate in the design of digital products such as apps, platforms, web apps, and AI tools to enable UNDP's work in countries, working with ‘client’ teams to define success and product requirements. 
Contribute to the scoping, and proposal writing for new digital products for UNDP thematic teams. 
Support the building of digital products, including working with software developers, client teams, and product managers. 
Assist with contract development and invoicing of product development work. 
Conduct user testing and improvement of digital products. 
Assist with the implementation of digital products in countries, working with local implementation teams and UNDP thematic teams to ensure quality delivery and outcomes.
 

Research & Knowledge Management 

Maintain and regularly update content on UNDP knowledge platforms (including SparkBlue and SharePoint) 
Conduct desk research and synthesis on digital, data, and AI topics to support KM products and communications 
Draft knowledge summaries, background notes, and briefing materials. 
Support content curation and dissemination of internal newsletters 
Curate and maintain digital- and data-related knowledge products and repositories 
Manage content for digital guides, internal platforms, and knowledge databases 
Coordinate dissemination of research and thought pieces across UNDP units 
 
 
 
COMPETENCIES
Achieve Results: Plans and monitors own work, pays attention to details, delivers quality work by deadline 
Think Innovatively: Open to creative ideas/known risks, is a pragmatic problem solver, makes improvements
Learn Continuously: Open minded and curious, shares knowledge, learns from mistakes, asks for feedback 
Adapt with Agility: Adapts to change, constructively handles ambiguity/uncertainty, is flexible 
Act with Determination:   Shows drive and motivation, able to deliver calmly in face of adversity, confident 
Engage and Partner: Demonstrates compassion/understanding towards others, forms positive relationships 
Enable Diversity and Inclusion: Appreciate/respect differences, aware of unconscious bias, confront discrimination 
 
OTHER COMPETENCIES
Database management skills.
Detail-oriented, preferably with previous administrative and office management experience.
Eager to learn new things and gradually adapt the theory into practical work.
Ability to work independently and collaboratively.
Excellent communication and interpersonal skills 
Ability to deliver in a high-pressure environment 
Ability to handle complex situations and multiple responsibilities simultaneously, mixing long-term projects with the urgency of immediate demands. 
Ability to collaborate with and achieve actionable results 
Excellent ability to multi-task in a high-demand and fast-paced work context 
Consistently approaches work with energy and a positive, constructive attitude 

 
 
REQUIRED SILLS AND EXPERIENCE

 
Education and Experience 

Candidates must meet one of the following educational requirements: 

Be enrolled in a graduate school programme (second university degree or equivalent, or higher), or. 
Be enrolled in the final academic year of a first university degree programme (undergraduate; minimum bachelor’s level or equivalent); or 
Have graduated with a university degree (first university degree, or second university degree or equivalent, or higher) and, if selected, must commence the internship within a one-year period of graduation. 
Interest/Experience in one of the workstream areas is desirable  
"""

In [10]:
prompt = f"""
You are a career assistant agent. You will be given a candidate's resume and a job posting.

Your tasks:
1. Write a tailored, professional cover letter (3-4 paragraphs) that connects the candidate's
   real skills/experience to this specific job posting. Do not invent skills or experience
   that aren't in the resume.
2. After the cover letter, add a section titled "SKILL MATCH" that lists:
   - Matched Skills: skills/experience from the resume that align with the job requirements
   - Gaps: requirements in the job posting that the resume doesn't clearly show

RESUME:
{resume_text}

JOB POSTING:
{job_posting_text}

Format your response with clear headers: "COVER LETTER" and "SKILL MATCH".
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

**COVER LETTER**

[Your Name - Mujeeb Ullah Khan]
[Your Address]
[Your Phone Number]
[Your Email Address]

[Date]

Hiring Team
Digital, AI and Innovation Internship: Global Call for 2026
United Nations Development Programme (UNDP)
Home Based

Dear Hiring Team,

I am writing to express my enthusiastic interest in the Digital, AI and Innovation Internship: Global Call for 2026 at the United Nations Development Programme (UNDP), as advertised. As a 6th-semester B.Sc. student in Information Systems, I am deeply passionate about leveraging technology, particularly Artificial Intelligence and the Internet of Things, to drive sustainable development and address global challenges. My academic background and hands-on project experience align perfectly with UNDP's mission to advance digital transformation and innovation for a better future, and I am eager to contribute my skills to your impactful work.

My resume highlights a strong foundation in core technologies critical to this internship, in

In [11]:
def generate_application_materials(resume_text, job_posting_text):
    """
    Takes a resume and job posting, returns a tailored cover letter + skill match.
    """
    prompt = f"""
You are a career assistant agent. You will be given a candidate's resume and a job posting.

Your tasks:
1. Write a tailored, professional cover letter (3-4 paragraphs) that connects the candidate's
   real skills/experience to this specific job posting. Do not invent skills or experience
   that aren't in the resume.
2. After the cover letter, add a section titled "SKILL MATCH" that lists:
   - Matched Skills: skills/experience from the resume that align with the job requirements
   - Gaps: requirements in the job posting that the resume doesn't clearly show

RESUME:
{resume_text}

JOB POSTING:
{job_posting_text}

Format your response with clear headers: "COVER LETTER" and "SKILL MATCH".
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [12]:
result = generate_application_materials(resume_text, job_posting_text)
print(result)

**COVER LETTER**

Dear Hiring Team,

I am writing to express my enthusiastic interest in the Digital, AI and Innovation Internship: Global Call for 2026 at the United Nations Development Programme (UNDP), as advertised on your platform. As a 6th-semester B.Sc. student in Information Systems at Raden Intan State Islamic University of Lampung, I am deeply passionate about leveraging technology, particularly Artificial Intelligence and the Internet of Things, to address complex global challenges. The UNDP’s commitment to advancing sustainable development through inclusive and responsible digital innovation perfectly aligns with my academic pursuits and professional aspirations.

My academic projects and practical experience have equipped me with a strong foundation in developing innovative digital solutions. I have hands-on experience in AI/ML, IoT systems, Python, JavaScript, HTML/CSS, and Firebase. My "IoT smart Door Project" demonstrates my ability to design and implement practical IoT

## Phase 3: Memory

In [13]:
import json
import os
from datetime import date

TRACKER_FILE = "/kaggle/working/applications.json"

def load_applications():
    if os.path.exists(TRACKER_FILE):
        with open(TRACKER_FILE, "r") as f:
            return json.load(f)
    return []

def save_applications(applications):
    with open(TRACKER_FILE, "w") as f:
        json.dump(applications, f, indent=2)

In [14]:
def log_application(company, role, status="Applied", notes=""):
    applications = load_applications()
    new_entry = {
        "company": company,
        "role": role,
        "date_applied": str(date.today()),
        "status": status,
        "notes": notes
    }
    applications.append(new_entry)
    save_applications(applications)
    print(f"Logged: {role} at {company} (status: {status})")

In [15]:
log_application(
    company="UNDP",
    role="Digital, AI and Innovation Internship",
    status="Applied",
    notes="Tailored cover letter generated via agent"
)

Logged: Digital, AI and Innovation Internship at UNDP (status: Applied)


In [16]:
def view_applications():
    applications = load_applications()
    if not applications:
        print("No applications logged yet.")
        return

    for i, app in enumerate(applications, start=1):
        print(f"{i}. {app['role']} at {app['company']}")
        print(f"   Status: {app['status']} | Applied: {app['date_applied']}")
        if app['notes']:
            print(f"   Notes: {app['notes']}")
        print()

In [17]:
view_applications()

1. Digital, AI and Innovation Internship at UNDP
   Status: Applied | Applied: 2026-06-29
   Notes: Tailored cover letter generated via agent



In [18]:
def update_application_status(company, role, new_status):
    applications = load_applications()
    updated = False

    for app in applications:
        if app['company'].lower() == company.lower() and app['role'].lower() == role.lower():
            app['status'] = new_status
            updated = True

    if updated:
        save_applications(applications)
        print(f"Updated: {role} at {company} → {new_status}")
    else:
        print(f"No matching application found for {role} at {company}")

In [19]:
update_application_status("UNDP", "Digital, AI and Innovation Internship", "Interview Scheduled")
view_applications()

Updated: Digital, AI and Innovation Internship at UNDP → Interview Scheduled
1. Digital, AI and Innovation Internship at UNDP
   Status: Interview Scheduled | Applied: 2026-06-29
   Notes: Tailored cover letter generated via agent



In [20]:
from google.genai import types

log_application_tool = {
    "name": "log_application",
    "description": "Log a new job/internship application the user has submitted.",
    "parameters": {
        "type": "object",
        "properties": {
            "company": {"type": "string", "description": "The company name"},
            "role": {"type": "string", "description": "The job/internship role/title"},
            "notes": {"type": "string", "description": "Any extra notes, optional"}
        },
        "required": ["company", "role"]
    }
}

update_status_tool = {
    "name": "update_application_status",
    "description": "Update the status of an existing application (e.g. Interview Scheduled, Rejected, Offer).",
    "parameters": {
        "type": "object",
        "properties": {
            "company": {"type": "string", "description": "The company name"},
            "role": {"type": "string", "description": "The job/internship role/title"},
            "new_status": {"type": "string", "description": "The new status, e.g. 'Interview Scheduled', 'Rejected', 'Offer'"}
        },
        "required": ["company", "role", "new_status"]
    }
}

view_applications_tool = {
    "name": "view_applications",
    "description": "Show all logged job/internship applications and their current statuses.",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}

tools = types.Tool(function_declarations=[
    log_application_tool,
    update_status_tool,
    view_applications_tool
])

In [21]:
chat_config = types.GenerateContentConfig(tools=[tools])

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=chat_config
)

## Phase 4: Natural Language Agent

In [22]:
def agent_respond(user_message):
    response = chat.send_message(user_message)

    part = response.candidates[0].content.parts[0]

    if part.function_call:
        fn_name = part.function_call.name
        fn_args = dict(part.function_call.args)

        print(f"[Agent decided to call: {fn_name} with {fn_args}]")

        if fn_name == "log_application":
            log_application(**fn_args)
        elif fn_name == "update_application_status":
            update_application_status(**fn_args)
        elif fn_name == "view_applications":
            view_applications()
        else:
            print("Unknown function requested.")
    else:
        print(response.text)

In [23]:
agent_respond("I just applied to Google for a Software Engineer role")

[Agent decided to call: log_application with {'role': 'Software Engineer', 'company': 'Google'}]
Logged: Software Engineer at Google (status: Applied)


In [24]:
agent_respond("Update my Google application to Interview Scheduled")

[Agent decided to call: update_application_status with {'company': 'Google', 'role': 'Software Engineer', 'new_status': 'Interview Scheduled'}]
Updated: Software Engineer at Google → Interview Scheduled


In [25]:
agent_respond("Show me all my applications")

[Agent decided to call: view_applications with {}]
1. Digital, AI and Innovation Internship at UNDP
   Status: Interview Scheduled | Applied: 2026-06-29
   Notes: Tailored cover letter generated via agent

2. Software Engineer at Google
   Status: Interview Scheduled | Applied: 2026-06-29

